# Agent Communication Protocol (ACP) | Agent Protocols

In [1]:
from dataclasses import dataclass
from typing import Callable, Dict, List

In [2]:
@dataclass
class ACPMessage:
    sender: str
    content: str
    msg_type: str = "text"  # text | question | report

class ACPRuntime:
    """Framework-agnostic runtime -- routes messages between any agent type."""
    def __init__(self):
        self._agents: Dict[str, Callable] = {}
        self._history: List[ACPMessage] = []

    def register(self, name: str, handler: Callable, framework: str = "unknown"):
        self._agents[name] = handler
        print(f"[ACP] Registered '{name}' (framework: {framework})")

    def send(self, sender: str, target: str, content: str, msg_type: str = "text") -> ACPMessage:
        msg = ACPMessage(sender=sender, content=content, msg_type=msg_type)
        self._history.append(msg)
        print(f"\n[ACP] {sender} -> {target}: [{msg_type}] {content[:80]}")
        reply_text = self._agents[target](msg)
        reply = ACPMessage(sender=target, content=reply_text)
        self._history.append(reply)
        print(f"[ACP] {target} -> {sender}: {reply.content[:80]}")
        return reply

In [3]:
# --- Agent A: Simple function (simulates a "LangChain-style" stateless agent) ---
def analyst_fn(msg: ACPMessage) -> str:
    if msg.msg_type == "question":
        return "Revenue split: Cloud 60%, On-prem 25%, Services 15%. Cloud grew 22% YoY."
    data = {"revenue": 2_400_000, "growth": 12.5, "top_segment": "Cloud"}
    return (f"Q3 Analysis: Revenue=${data['revenue']:,}, Growth={data['growth']}%, "
            f"Top={data['top_segment']}. Ask me for segment breakdown if needed.")

# --- Agent B: Class-based (simulates a "CrewAI-style" agent with memory) ---
class ReportAgent:
    def __init__(self):
        self.gathered: List[str] = []

    def handle(self, msg: ACPMessage) -> str:
        self.gathered.append(msg.content)
        if len(self.gathered) == 1:  # first message -- need more detail
            return "QUESTION: Can you break down revenue by segment with YoY growth?"
        return (f"REPORT: Q3 Executive Summary\n"
                f"  Overview: {self.gathered[0]}\n"
                f"  Detail: {self.gathered[1]}\n"
                f"  Recommendation: Double down on Cloud segment (22% YoY growth).")

reporter = ReportAgent()

In [4]:
# --- Wire both into ACP runtime ---
runtime = ACPRuntime()
runtime.register("analyst", analyst_fn, framework="function-based")
runtime.register("reporter", reporter.handle, framework="class-based")

# Multi-step conversation (4 exchanges demonstrating interoperability):
# Step 1: Reporter asks Analyst for initial analysis
r1 = runtime.send("reporter", "analyst", "Analyze Q3 sales data")

# Step 2: Analyst's response goes to Reporter -- Reporter asks clarifying question
r2 = runtime.send("analyst", "reporter", r1.content)

# Step 3: Reporter's question goes back to Analyst for segment detail
r3 = runtime.send("reporter", "analyst", r2.content, msg_type="question")

# Step 4: Analyst's detail goes to Reporter -- Reporter produces final report
r4 = runtime.send("analyst", "reporter", r3.content)

print(f"\n{'='*60}")
print(f"Final output:\n{r4.content}")
print(f"\nACP routed {len(runtime._history)} messages between 2 framework types")

[ACP] Registered 'analyst' (framework: function-based)
[ACP] Registered 'reporter' (framework: class-based)

[ACP] reporter -> analyst: [text] Analyze Q3 sales data
[ACP] analyst -> reporter: Q3 Analysis: Revenue=$2,400,000, Growth=12.5%, Top=Cloud. Ask me for segment bre

[ACP] analyst -> reporter: [text] Q3 Analysis: Revenue=$2,400,000, Growth=12.5%, Top=Cloud. Ask me for segment bre
[ACP] reporter -> analyst: QUESTION: Can you break down revenue by segment with YoY growth?

[ACP] reporter -> analyst: [question] QUESTION: Can you break down revenue by segment with YoY growth?
[ACP] analyst -> reporter: Revenue split: Cloud 60%, On-prem 25%, Services 15%. Cloud grew 22% YoY.

[ACP] analyst -> reporter: [text] Revenue split: Cloud 60%, On-prem 25%, Services 15%. Cloud grew 22% YoY.
[ACP] reporter -> analyst: REPORT: Q3 Executive Summary
  Overview: Q3 Analysis: Revenue=$2,400,000, Growth

Final output:
REPORT: Q3 Executive Summary
  Overview: Q3 Analysis: Revenue=$2,400,000, Growth=12.